<div style="display: flex; align-items: center;">
  <img src="../Images/Logos/DecisionIntelligenceWorkshopLogo.png" width="60px" style="margin-right: 10px">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Decisions with Chat Completions & Responses APIs</span>
</div>

Decision Intelligence applied in this module:  
* Introduces the OpenAI Chat Completions API & Responses API as LLM AI interfaces 
* Chat Completions: Composing decisions with system and instruction prompts. Listing of various decision-making frameworks and with their descriptions 
* Chat Completions: Crafting a multi-turn decision scenario 
* Chat Completions: Inspecting the gathered intelligence (chat history)


OpenAI offers two main APIs for building with its models: **Chat Completions** and the **Responses API**. Chat Completions is the established interface for sending conversational messages and receiving model-generated replies, making it useful for chatbots, assistants, and straightforward text generation workflows. The Responses API is the newer, more flexible interface designed to support modern AI applications, including multimodal inputs, structured outputs, tool use, and stateful interactions.

**Chat Completions commonly provides:**

* Message-based chat interactions using roles like system, user, and assistant
* Stateless requests by default; your application sends prior messages each turn
* Full control over what conversation history and context the model receives
* Broad compatibility with existing OpenAI chat-based implementations

**Responses API commonly provides:**

* A unified interface for text, multimodal inputs, tools, and structured outputs
* Built-in stateful context using store: true and prior response chaining
* Easier multi-turn interactions without always resending the full message history
* Better support for deeper reasoning models (GPT-5.4+) 

In this module, the Microsoft Extensions for AI (MEAI) ability to create a Chat experience will be introduced. This is a much richer experience than just sending simple prompts that are stateless and context is forgotten in subsequent requests.

MEAI has first-class support for chat scenarios, where the user talks back and forth with the LLM, the arguments get populated with the history of the conversation. During each new run of the decision chats, the arguments will be provided to the AI with content. This allows the LLM to know the historical context of the conversation.

----
### Step 1 - Initialize Configuration Builder & Build the AI Orchestration

Execute the next two cells to:
* Use the Configuration Builder to load the API secrets.  
* Use the API configuration to build the ChatCompletions orchestrator. 

In [1]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 10.0.9"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.9"
#r "nuget: System.Text.Json, 10.0.9"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.9 Microsoft.Extensions.Configuration.Json, 10.0.9 System.Text.Json, 10.0.9

In [2]:
// Import the Microdoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.7.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.7.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.7.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.12.0"

using Azure;
using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient based on the Azure OpenAI Service or OpenAI Service depending on the flag
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    chatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    chatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();
}

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.7.0 Microsoft.Extensions.AI.Abstractions, 10.7.0 Microsoft.Extensions.AI.OpenAI, 10.7.0 OpenAI, 2.12.0

Using Azure OpenAI Service



warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.12.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.14.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.14.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'OpenAI, Ve

### Step 2 - Chat Completions: Decisions with Prompt Execution Settings 

<img style="display: block; margin: auto;" width ="600px" src="../Images/DecisionIntelligence-Framework-DemingQuote.png">


Using the Microsoft Extensions AI ChatCompletion service is very similar to inovoking a prompt for basic LLM interactions. The chat completion service will provide very similar results to invoking the prompt directly.

In [3]:
// Simple prompt to list some decision frameworks this GenAI LLM is familiar with 
// LLMs are trained on a diverse range of data and can provide insights on a wide range of topics like decision frameworks
// SLMS (smaller LLMs) are trained on a more specific range of data and may not provide insights on all topics
var simpleDecisionPrompt = """
Provide a list of 5 decision frameworks that can help improve the quality of decisions.

Output Format Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.
""";

// Execute the prompt against the AI model
var simpleDecisionPromptResponse = await chatClient.GetResponseAsync(simpleDecisionPrompt);
var simpleDecisionPromptResponseText = simpleDecisionPromptResponse.Text;

// Display the response string as Markdown
simpleDecisionPromptResponseText.DisplayAs("text/markdown");

### 5 Decision Frameworks

1. **SWOT Analysis**  
   Evaluate a decision by identifying its **Strengths, Weaknesses, Opportunities, and Threats**. This helps clarify internal capabilities and external risks.

2. **Decision Matrix**  
   List possible options, define evaluation criteria, assign weights to those criteria, and score each option. This provides a structured way to compare alternatives.

3. **Cost–Benefit Analysis**  
   Compare the expected benefits of each option with its financial, operational, and opportunity costs. Choose the option that offers the strongest overall value.

4. **Six Thinking Hats**  
   Examine a decision from six perspectives: facts, emotions, risks, benefits, creativity, and process. This reduces bias and encourages balanced thinking.

5. **Pre-mortem Analysis**  
   Imagine that the decision has failed, then identify the reasons why. This surfaces hidden risks and allows you to address weaknesses before committing.

----
### Step 3 - Chat Completions: Decision Chat with a System Prompt & Chat History 

In the previous examples, simple decision prompts were used. In more sophisticated scenarios, a conversational history and state is required to be maintained. The Chat Completions API definition includes mechanisms to maintain state.  

Microsoft Extensions for AI includes a ChatHistory object that can be used with the ChatCompletionService to provide historical chat context to the LLM. Notice that the ChatHistory object differentiates between the different types of chat messages:
* System Message - System or MetaPrompt. These are usually global instructions that set the "overall rules" for interacting with the LLMs.
* User Message - A message from the user
* Assistant Message - A message from the LLM. This is a message generated from an assistant or an agent. 

Identifying the messages from which role (user) it came from can help the LLM improve its own reasoning and decision responses. This is a more sophisticated approach than passing chat history in a long dynamic string. ChatHistory objects can be serialized and persisted into databases as well. This allows a system architect to load chat history dynamically, branch history conversations, re-run or simulate different conversation paths. For decision scenarios, these tools are super helpful. 

In [4]:
// Set the overall system prompt to behave like a decision intelligence assistant (persona)
var systemPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Format Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Simple instruction prompt to list 5 (five) decision frameworks this GenAI LLM is familiar with
var simpleDecisionPrompt = """
Provide five Decision Frameworks that can help improve the quality of decisions.
""";

// Create a new chat history object with proper system and user message roles
var chatMessages = new List<ChatMessage>();
// Add system and user messages to the chat history
var systemMessage = new ChatMessage(ChatRole.System, systemPrompt);
var userMessage = new ChatMessage(ChatRole.User, simpleDecisionPrompt);
chatMessages.Add(systemMessage);
chatMessages.Add(userMessage);


// Execute the chat messages against the AI model
var chatHistoryResponse = await chatClient.GetResponseAsync(chatMessages);
var chatHistoryResponseText = chatHistoryResponse.Text;

// Display the response string as Markdown
chatHistoryResponseText.DisplayAs("text/markdown");

// Capture the Assistant response and add it to the chat history
// This will persist the full conversation for future interactions
var assistantChatMessage = new ChatMessage(ChatRole.Assistant, chatHistoryResponseText);

| # | Decision framework | How it improves decision quality | Best used when | Key steps | Common pitfall |
|---:|---|---|---|---|---|
| 1 | **Weighted Decision Matrix** | Makes evaluation criteria explicit and compares options consistently rather than relying on intuition alone. | Choosing among multiple options with different trade-offs, such as vendors, jobs, projects, or products. | Define criteria → assign importance weights → score each option → calculate weighted totals → test sensitivity to different weights. | False precision: numerical scores can hide uncertain assumptions or subjective judgments. |
| 2 | **Expected Value Analysis** | Incorporates both the likelihood and impact of possible outcomes, helping prioritize choices under uncertainty. | Decisions involving risk, probabilities, financial outcomes, or multiple possible scenarios. | List possible outcomes → estimate probabilities → assign values or costs → calculate probability × outcome → compare expected values. | Overconfidence in probability estimates can make the result misleading. |
| 3 | **Pre-Mortem Analysis** | Reduces overconfidence and exposes weaknesses by imagining that a decision has already failed. | Major investments, strategic plans, launches, hiring decisions, or other high-consequence choices. | Assume the decision failed → identify plausible causes → rank risks → add mitigations → revise the decision or implementation plan. | Teams may focus only on dramatic risks and overlook ordinary execution problems. |
| 4 | **OODA Loop** | Encourages rapid learning and adaptation by treating decisions as an iterative cycle rather than a one-time event. | Fast-changing environments, competitive situations, operations, emergencies, or experiments. | Observe current conditions → orient using context and assumptions → decide → act → observe the results and repeat. | Acting too quickly without adequate observation or reflection can reinforce poor assumptions. |
| 5 | **Reversible vs. Irreversible Decision Framework** | Prevents over-analysis of low-risk choices while ensuring sufficient diligence for decisions that are difficult or costly to undo. | Managing decision speed, resource allocation, product experimentation, organizational changes, or strategic commitments. | Determine whether the decision is reversible → set an appropriate analysis threshold → act quickly on reversible choices → use deeper review for irreversible ones. | A decision may appear reversible but create hidden reputational, legal, or opportunity costs. |

In the next step, an additional prompt instruction will be added to the chat history. From the 5 decision frameworks provided in the chat history, Generative AI is asked to recommed a decision framework best suited for the militaty intelligence community. Notice that the previous chat history is automatically provided to provide additional intelligence context. 

<img style="display: block; margin: auto;" width ="600px" src="../Images/DecisionIntelligence-DecisionExecution-NapoleonQuote.png">

In [5]:
// Add the assistant message to the chat history
chatMessages.Add(assistantChatMessage);

// Note: No reference is made to what previous frameworks were listed.
// Note: Previous context is maintained by the MEAI ChatHistory object 
var simpleDecisionPromptFollowupQuestionPartTwo = """
Which of the 5 decision frameworks listed above is best suited for the military intelligence community?  
Think carefully step by step about what decision frameworks are needed to answer the query.  
Select only the single best framework. 
""";

// Add User message to the chat history
var userFollowupChatMessage = new ChatMessage(ChatRole.User, simpleDecisionPromptFollowupQuestionPartTwo);
chatMessages.Add(userFollowupChatMessage);

// Execute the chat messages against the AI model
var chatHistoryResponseMilitaryIntelligence = await chatClient.GetResponseAsync(chatMessages);
var chatHistoryResponseMilitaryIntelligenceText = chatHistoryResponseMilitaryIntelligence.Text;

// Display the response string as Markdown
chatHistoryResponseMilitaryIntelligenceText.DisplayAs("text/markdown");

| Selection | Best framework for the military intelligence community | Why it is the strongest fit | How it applies |
|---|---|---|---|
| **1** | **OODA Loop — Observe, Orient, Decide, Act** | Military intelligence operates in adversarial, rapidly changing environments where information is incomplete, deceptive, and time-sensitive. OODA directly supports continuous sensing, interpretation, decision-making, action, and reassessment. Its emphasis on outlearning and outmaneuvering an opponent makes it more suitable than frameworks focused primarily on static scoring, expected-value calculations, or one-time risk reviews. | **Observe:** collect intelligence and detect changes. **Orient:** assess context, adversary intent, deception, uncertainty, and source reliability. **Decide:** select an intelligence or operational response. **Act:** disseminate assessments, task collection, or support operations. Then repeat the cycle as new information arrives. |
| **Why not the others as the single best choice** | Weighted Decision Matrix, Expected Value Analysis, Pre-Mortem Analysis, and Reversible vs. Irreversible Decisions remain useful supporting tools, but they are narrower. | They help compare options, quantify risk, identify failure modes, or calibrate decision speed; however, they do not provide as complete an operating cycle for dynamic intelligence competition. | Use them within the OODA cycle—for example, expected value for uncertain courses of action and pre-mortems for major assessments—while keeping OODA as the primary framework. |

-----
### Step 4 - Chat Completions: Decision Conversation with a Multi-Turn Conversation Scenario 

<img style="display: block; margin: auto;" width ="600px" src="../Images/DecisionIntelligence-Quote-WarrenBuffet-Value.png"> 

The list of Chat Messages persists the state. This allows for natural converations between the user and AI to be persisted for future use. It can be saved for future reference, it can be replayed, it can be forked in certain spots to try different scenarios, it can be used as a starting point for other conversations etc. 

In the scenario below, let's build a multi-turn conversation about a decision over a purchase for a fitness tracker. Notice that each turn the appropriate Chat History roles are populated with the appropriate User prompts and assistant (AI) responses. This in effect acts as the Gathering Intelligence step of the Decision Intelligence framework as the conversation information has been persisted and can be re-used as context for the decision scenario.

In [6]:
// Set the overall system prompt to behave like a decision intelligence assistant (persona)
var systemPromptDecisionConversation = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.
""";

// User prompt to start a decision conversation about buying a fitness tracker
var userPromptDecisionConversationTurn1 = """
I am considering buying a fitness tracker but I am not sure which features really matter. 
Could you explain the main considerations? 
""";

var assistantResponseDecisionConversationTurn1 = """
Absolutely! Key considerations for choosing a fitness tracker often include: 
- Fitness Features (e.g., step counting, heart-rate monitoring, workout tracking)
- Health Monitoring (like sleep tracking or advanced ECG capabilities)
- Battery Life
- Comfort and Fit
- Compatibility with your smartphone's operating system
- Price Range
- Knowing which of these you value most will help narrow down your options.
""";

var userPromptDecisionConversationTurn2 = """
Got it. Battery life and sleep tracking are both important to me. 
What's a typical battery life range for trackers, and how good is their sleep tracking?
""";

var assistantResponseDecisionConversationTurn2 = """
Different fitness trackers have varying battery lives, typically anywhere from 2-3 days (for more feature-rich or bright-display devices) 
up to about 10-14 days (for more minimalist trackers). 

Regarding sleep tracking:
- Many modern devices can track different sleep stages (light, deep, REM).
- High-end devices even track breathing rate and oxygen saturation during the night.
- Some also offer insights about sleep quality and tips for improvement.

If battery life is a priority, you may want to consider trackers with a more conservative display and fewer power-hungry features.
""";

var userPromptDecisionConversationTurn3 = """
Are there any popular models you would recommend for that balance of battery life and sleep tracking?
""";

var assistantResponseDecisionConversationTurn3 = """
Popular options that strike a balance between robust sleep tracking and good battery life include:
- Fitbit Inspire 3 or Fitbit Charge 5: Both have solid sleep tracking, multi-day battery, and strong app support.
- Garmin Vivosmart 5: Offers decent fitness and sleep insights, plus better-than-average battery life.
- Xiaomi Mi Band 7: Budget-friendly, surprisingly robust health metrics, and a long-lasting battery.

Each varies in price and features, so consider your budget and the specific metrics you’d like to track.
""";

var userPromptDecisionConversationTurn4 = """
That is great infor. 
I also have a friend who said I should consider an Apple Watch. 
Does that have comparable battery life and sleep tracking?
""";

var assistantResponseDecisionConversationTurn4 = """
The Apple Watch is known for its extensive feature set, including advanced health monitoring, 
but its battery life tends to be shorter—usually around 18-24 hours on a single charge 
(or possibly a bit more in low-power mode). While its sleep tracking features have improved over time, 
you might find you need to charge it daily or almost daily, which could conflict with your goal of 24/7 tracking. 
""";

// Create a new chat history with a multi-turn conversation about buying a fitness tracker
// Notice how the conversation flow uses different roles (user and assistant) to simulate a realistic decision-making dialogue
var chatHistoryMessages = new List<ChatMessage>();

chatHistoryMessages.Add(new ChatMessage(ChatRole.System, systemPromptDecisionConversation));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn1));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn1));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn2));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn2));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn3));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn3));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn4));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn4));


Using the above conversation history as the "Gathered Intelligence" betweeen the user and the AI assistant, let's make a final decision and ask for final feedback and a decision evaluation. This "Gathered Intelligence" for the decision can be persisted, re-loaded, simulated multiple times, applied to different AI systems to optimize decisions further.  

In [7]:
var userPromptDecisionConversationFinalDecision = """
Thank you for all of that information, I think I'll go with a Fitbit. 
The Fitbit Inspire 3 seems like a good fit for my budget and needs. 

Any quick final thoughts or insights on my decision? 
""";

// Add User message to the chat history
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationFinalDecision));

// Execute the chat messages against the AI model
var fitnessTrackerResponse = await chatClient.GetResponseAsync(chatHistoryMessages);
var fitnessTrackerResponseText = fitnessTrackerResponse.Text;

// Add the response to the chat history (Chat Messages)
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, fitnessTrackerResponseText));

// Display the response string as Markdown
fitnessTrackerResponseText.DisplayAs("text/markdown");

The Fitbit Inspire 3 sounds like a sensible choice for your priorities:

- **Battery:** Up to about **10 days** under typical conditions, though frequent exercise tracking, notifications, and always-on features can reduce this.
- **Sleep tracking:** Good for monitoring sleep duration, consistency, and estimated stages. Treat sleep-stage and “sleep score” results as trends rather than medical measurements.
- **Comfort:** Its small, lightweight design is well suited to overnight wear.
- **Trade-offs:** It lacks built-in GPS, so it generally uses your phone for route tracking. Some advanced insights and coaching features require a **Fitbit Premium** subscription.
- **Practical tip:** Wear it snugly but comfortably, keep the sensor clean, and establish a charging routine that does not interrupt your sleep—such as charging while showering.

Overall, it is a good budget-friendly fit if you value long battery life and basic-to-moderate health tracking more than smartwatch apps or advanced workout features.

----
### Step 5 - Chat Completions: Inspect & Optimize Gathered Intelligence of Chat Completion History 

The ChatMessage list is a transparent construct that can be inspected and written out. Because it is a simple list, the chat messages object can be manipulated to replay chats from middle interactions to simulate different outcomes.

Execute the cell below to write out entire decision conversation. 

> **📝 Note:** In the Decision Intelligence framework, chat history can serve as a form of gathered intelligence. Interactions among users, AI models, processes, and agents provide valuable context for effective decision-making. It’s highly recommended to persist chat history objects (agent threads), especially during decision optimization, to ensure critical information is retained and accessible.  

In [8]:
// Print the number of chat interactions and the chat history (turns)
Console.WriteLine("Number of chat interactions: " + chatHistoryMessages.Count());

// Change this to a string builder and show as markdown
var stringBuilderChatHistory = new StringBuilder();
foreach (var message in chatHistoryMessages)
{
    // add a new line for each message
    stringBuilderChatHistory.AppendLine($"**{message.Role.ToString().ToUpper()}**:");
    stringBuilderChatHistory.Append($"{message.Text.Replace("#", string.Empty)}");
    stringBuilderChatHistory.AppendLine("\n");
}

// Display the chat history as Markdown
stringBuilderChatHistory.ToString().DisplayAs("text/markdown");

Number of chat interactions: 11


**SYSTEM**:
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

**USER**:
I am considering buying a fitness tracker but I am not sure which features really matter. 
Could you explain the main considerations? 

**ASSISTANT**:
Absolutely! Key considerations for choosing a fitness tracker often include: 
- Fitness Features (e.g., step counting, heart-rate monitoring, workout tracking)
- Health Monitoring (like sleep tracking or advanced ECG capabilities)
- Battery Life
- Comfort and Fit
- Compatibility with your smartphone's operating system
- Price Range
- Knowing which of these you value most will help narrow down your options.

**USER**:
Got it. Battery life and sleep tracking are both important to me. 
What's a typical battery life range for trackers, and how good is their sleep tracking?

**ASSISTANT**:
Different fitness trackers have varying battery lives, typically anywhere from 2-3 days (for more feature-rich or bright-display devices) 
up to about 10-14 days (for more minimalist trackers). 

Regarding sleep tracking:
- Many modern devices can track different sleep stages (light, deep, REM).
- High-end devices even track breathing rate and oxygen saturation during the night.
- Some also offer insights about sleep quality and tips for improvement.

If battery life is a priority, you may want to consider trackers with a more conservative display and fewer power-hungry features.

**USER**:
Are there any popular models you would recommend for that balance of battery life and sleep tracking?

**ASSISTANT**:
Popular options that strike a balance between robust sleep tracking and good battery life include:
- Fitbit Inspire 3 or Fitbit Charge 5: Both have solid sleep tracking, multi-day battery, and strong app support.
- Garmin Vivosmart 5: Offers decent fitness and sleep insights, plus better-than-average battery life.
- Xiaomi Mi Band 7: Budget-friendly, surprisingly robust health metrics, and a long-lasting battery.

Each varies in price and features, so consider your budget and the specific metrics you’d like to track.

**USER**:
That is great infor. 
I also have a friend who said I should consider an Apple Watch. 
Does that have comparable battery life and sleep tracking?

**ASSISTANT**:
The Apple Watch is known for its extensive feature set, including advanced health monitoring, 
but its battery life tends to be shorter—usually around 18-24 hours on a single charge 
(or possibly a bit more in low-power mode). While its sleep tracking features have improved over time, 
you might find you need to charge it daily or almost daily, which could conflict with your goal of 24/7 tracking. 

**USER**:
Thank you for all of that information, I think I'll go with a Fitbit. 
The Fitbit Inspire 3 seems like a good fit for my budget and needs. 

Any quick final thoughts or insights on my decision? 

**ASSISTANT**:
The Fitbit Inspire 3 sounds like a sensible choice for your priorities:

- **Battery:** Up to about **10 days** under typical conditions, though frequent exercise tracking, notifications, and always-on features can reduce this.
- **Sleep tracking:** Good for monitoring sleep duration, consistency, and estimated stages. Treat sleep-stage and “sleep score” results as trends rather than medical measurements.
- **Comfort:** Its small, lightweight design is well suited to overnight wear.
- **Trade-offs:** It lacks built-in GPS, so it generally uses your phone for route tracking. Some advanced insights and coaching features require a **Fitbit Premium** subscription.
- **Practical tip:** Wear it snugly but comfortably, keep the sensor clean, and establish a charging routine that does not interrupt your sleep—such as charging while showering.

Overall, it is a good budget-friendly fit if you value long battery life and basic-to-moderate health tracking more than smartwatch apps or advanced workout features.



By persisting the Decision Intelligence chats and making them available for inspection, can further help the decision-making process. For example you can load the Chat History summarize it or even have AI recommend optimizations for it based on the conversation. 

> **📝 Note:** The process below is simplified to illustrate the power of Gathering Intelligence with user <--> AI interactions. More advanced best practices will be introduced in further sections. 

In [9]:
var userPromptSummarizeTheDecisionChat = """
Briefly summarize the chat conversation about buying a fitness tracker.
""";

chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptSummarizeTheDecisionChat));

// Execute the chat messages against the AI model
var fitnessTrackerDecisionSummaryResponse = await chatClient.GetResponseAsync(chatHistoryMessages);
var fitnessTrackerDecisionSummaryResponseText = fitnessTrackerDecisionSummaryResponse.Text;

// Display the response string as Markdown
fitnessTrackerDecisionSummaryResponseText.DisplayAs("text/markdown");

You prioritized **long battery life** and **reliable sleep tracking** when choosing a fitness tracker. We discussed typical battery ranges of roughly **2–3 days to 10–14 days**, and noted that sleep tracking generally estimates sleep duration and stages, with some devices offering breathing and blood-oxygen metrics.

We compared options including Fitbit, Garmin, Xiaomi, and Apple Watch. The Apple Watch offers broader features but usually needs **daily charging**, so you decided the **Fitbit Inspire 3** best fits your budget and needs. Its strengths are approximately **10-day battery life**, lightweight overnight comfort, and useful sleep trends; trade-offs include no built-in GPS and some optional Fitbit Premium features.

The chat history messsage can further be manipulated by looking at the user to AI interactions and getting feedback for future decision interactions. 

In [10]:
// Optimize the chat history for any future decision interactions
var userPromptOptimizeTheDecisionChat = """
Based on the chat conversation so far...
Provide some recommendations on how the decision interactions could be optimized for decision-making.
What are some considerations for the AI to help the use make even better decisions?
""";

// Remove the last message to avoid redundancy
chatHistoryMessages.RemoveAt(chatHistoryMessages.Count - 1);

chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptOptimizeTheDecisionChat));

// Execute the chat messages against the AI model
var decisionOptimizationResponse = await chatClient.GetResponseAsync(chatHistoryMessages);
var decisionOptimizationResponseText = decisionOptimizationResponse.Text;

// Display the response string as Markdown
decisionOptimizationResponseText.DisplayAs("text/markdown");

The conversation was helpful and reached a plausible recommendation, but the decision process could be made more rigorous and personalized.

## What could be improved

### 1. Ask clarifying questions earlier
The assistant moved to model recommendations before establishing several important constraints:

- **Phone:** iPhone or Android? Fitbit compatibility and features can vary.
- **Budget:** Purchase price and willingness to pay for Fitbit Premium.
- **Exercise needs:** Casual activity, gym workouts, running, cycling, swimming?
- **GPS:** Is built-in GPS important, or is carrying a phone acceptable?
- **Sleep priorities:** Basic duration and consistency, or detailed trends, breathing disturbances, and recovery metrics?
- **Health requirements:** Are ECG, irregular-rhythm alerts, or blood-oxygen trends relevant?
- **Comfort:** Wrist size, sensitivity to wearing a device overnight, band preferences.
- **Smartwatch features:** Notifications, payments, music controls, calls, apps.
- **Data preferences:** Comfort with cloud accounts, health-data sharing, and subscription services.

A few targeted questions would have prevented the recommendation from relying on assumptions.

### 2. Separate “must-have” from “nice-to-have” features
The user identified battery life and sleep tracking as priorities, but the assistant could have helped rank them explicitly:

| Criterion | Importance | Minimum acceptable level |
|---|---:|---|
| Battery life | High | At least 5–7 days |
| Overnight comfort | High | Lightweight, comfortable band |
| Sleep tracking | High | Sleep duration and trend tracking |
| GPS | Unknown | Built-in, phone-connected, or unnecessary |
| Price | Unknown | Set a maximum |
| Subscription | Unknown | Acceptable or not acceptable |

This makes the decision less vulnerable to attractive but irrelevant features.

### 3. Use current, model-specific information
The discussion included broad estimates, but product specifications can change with firmware, settings, availability, and region. A stronger answer would verify:

- Current battery estimate and charging time
- Whether all desired sleep features are available without Premium
- Phone compatibility
- Warranty and return policy
- Current retail price
- Whether the model is still actively supported

For example, “up to 10 days” should be presented as a manufacturer estimate, not a guaranteed real-world result. Frequent workouts, notifications, blood-oxygen measurements, and display use can reduce it.

### 4. Be more precise about sleep-tracking quality
Sleep tracking should be framed appropriately:

- Sleep duration and approximate sleep/wake timing are generally more useful than exact sleep-stage results.
- Consumer trackers estimate sleep stages using movement and optical signals; they do not measure brain waves like a clinical sleep study.
- Trends over multiple nights are more meaningful than a single “sleep score.”
- Sleep metrics should not be used to diagnose sleep disorders.

The assistant should also clarify what the user actually wants from sleep tracking: awareness, habit improvement, recovery guidance, or medical concern.

### 5. Present alternatives and trade-offs fairly
The conversation quickly converged on Fitbit Inspire 3. A better approach would compare a small set of finalists based on the user’s priorities:

- **Fitbit Inspire 3:** Strong battery and comfortable sleep wear; limited smartwatch and GPS functionality.
- **Garmin vívosmart or similar Garmin tracker:** Often good battery and fitness ecosystem; sleep insights and app experience may differ.
- **Xiaomi/other budget bands:** Long battery and low price; app quality, privacy, support, and sleep interpretation may be less attractive depending on region.
- **Apple Watch:** Better smartwatch and health ecosystem, but generally much weaker battery fit for someone prioritizing uninterrupted overnight tracking.

This gives the user a clear reason for choosing Fitbit rather than simply confirming their initial preference.

### 6. Account for total cost and lock-in
The recommendation should include:

- Device cost
- Optional subscription cost over one or two years
- Replacement bands and charger
- Warranty and expected lifespan
- Whether important data or insights are locked behind a subscription
- Ease of exporting data if the user later changes platforms

A cheap device may not remain the cheapest choice if essential features require a recurring fee.

### 7. Communicate uncertainty and recommendation confidence
Rather than saying it is simply “a sensible choice,” the assistant could say:

> “Given your stated priorities—long battery life, overnight comfort, and basic sleep trends—the Inspire 3 appears to be a good fit, assuming you do not need built-in GPS, advanced workout features, or a full smartwatch.”

That makes the conditions behind the recommendation explicit.

## A stronger decision interaction

An optimized interaction could follow this sequence:

1. **Elicit priorities and constraints.**
2. **Clarify what “good sleep tracking” means to the user.**
3. **Set minimum requirements and deal-breakers.**
4. **Compare three or four suitable models.**
5. **Include total cost, subscriptions, privacy, and support.**
6. **Recommend one option conditionally.**
7. **Suggest a return-policy or trial-based validation plan.**

For example, the AI might ask:

> “Before choosing the Inspire 3, do you use an iPhone or Android, do you need built-in GPS, and are you willing to pay for Fitbit Premium? Also, is your goal basic sleep-duration tracking or more detailed recovery and health insights?”

## Practical final check for this decision

The user should confirm these points before buying:

- It works fully with their phone.
- The expected battery meets their minimum in real use.
- The desired sleep features are available without an unwanted subscription.
- They do not need built-in GPS or smartwatch functions.
- The band is comfortable enough for overnight wear.
- The retailer offers a reasonable return period.
- The device’s privacy and account requirements are acceptable.

Overall, the recommendation was directionally reasonable, but the AI should have done more preference elicitation, current fact-checking, trade-off analysis, and uncertainty disclosure before endorsing the specific model.